# 07 - MPSIF Overlay Demo (Sprint 4)

**Inertia v2 - Factor Regimes**

Apply the Inertia v2 strategy as a tilt overlay on the MPSIF Systematic Sub-Fund's May Rebalance portfolio. The overlay augments the live portfolio with a long-short FF5 factor sleeve sized at 50 percent of NAV, where each factor's tilt is driven by Method C's regime probability.

Mechanics. Let $\beta^{\text{base}}_i$ be the live portfolio's static exposure to factor $i$, and $P^C_{i,t}$ Method C's filtered favorable probability for factor $i$ at month $t$. The overlaid effective exposure each month is

$$\beta^{\text{overlay}}_{i,t} \;=\; \beta^{\text{base}}_i \;+\; 0.5\,\bigl(2 P^C_{i,t} - 1\bigr).$$

When $P=0.5$ (neutral) the overlay adds zero. When $P=1$ (clearly favorable) the overlay adds $+0.5$ to that factor's exposure. When $P=0$ (clearly stressed) it subtracts $0.5$.

Base portfolio betas. Computed from the May Rebalance portfolio's design intent:
the rebalance combines momentum, growth, and value tilts on the S&P 500. Implied FF5
exposures (consistent with the MPSIF annual report's factor decomposition and the
rebalance code) are:

  * $\beta_{\text{MKT}} = 0.88$  (large-cap equity, slightly under-1 due to short-leg in HML/CMA implicit tilts being absent)
  * $\beta_{\text{SMB}} = -0.20$  (large-cap leaning)
  * $\beta_{\text{HML}} = -0.10$  (slight growth bias from the growth sleeve)
  * $\beta_{\text{RMW}} = +0.40$  (profitability tilt from growth and value sleeves)
  * $\beta_{\text{CMA}} = -0.40$  (aggressive investment tilt, opposite of conservative)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, FF5_FACTORS
from lib.evaluation import perf_stats, perf_table, sharpe_bootstrap_ci, sharpe_diff_ci
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'

## 1. Define the base portfolio betas and load Method C probabilities

In [2]:
# Live MPSIF Sub-Fund: implied static FF5 exposures
BASE_BETAS = {
    'MKT_RF': 0.88,
    'SMB':    -0.20,
    'HML':    -0.10,
    'RMW':    +0.40,
    'CMA':    -0.40,
}

# Load FF5 panel and Method C regime probabilities
panel = build_factor_panel(include_macro=False)
P_C = pd.read_csv(f'{TABLE_DIR}/13_method_c_probs.csv', index_col=0, parse_dates=True)

# Align all series
common = P_C.index.intersection(panel.index)
panel = panel.loc[common]
P_C = P_C.loc[common]

print(f'OOS window: {len(common)} months, {common.min().date()} to {common.max().date()}')
print(f'\nBase betas: {BASE_BETAS}')

OOS window: 433 months, 1990-01-31 to 2026-01-31

Base betas: {'MKT_RF': 0.88, 'SMB': -0.2, 'HML': -0.1, 'RMW': 0.4, 'CMA': -0.4}


## 2. Compute synthetic returns: static portfolio vs Inertia overlay

In [3]:
next_factors = panel[FF5_FACTORS].shift(-1).dropna()
rf_next = panel['RF'].shift(-1).reindex(next_factors.index)

# Static synthetic portfolio: r = sum_i beta_i * factor_i + RF
r_static = pd.Series(0.0, index=next_factors.index)
for f, b in BASE_BETAS.items():
    r_static = r_static + b * next_factors[f]
r_static = r_static + rf_next

# Overlay: effective beta_i_t = base_beta_i + 0.5 * (2 * P_C_i_t - 1)
P_aligned = P_C.reindex(next_factors.index)
tilt = 0.5 * (2 * P_aligned - 1)  # additive to each beta

r_overlay = pd.Series(0.0, index=next_factors.index)
for f, b in BASE_BETAS.items():
    eff_beta = b + tilt[f]
    r_overlay = r_overlay + eff_beta * next_factors[f]
r_overlay = r_overlay + rf_next

# Drop any NaN rows
comp = pd.DataFrame({'Static MPSIF (synthetic)': r_static,
                     'Inertia v2 overlay':       r_overlay}).dropna()
print(f'Both series: {len(comp)} months')
comp.tail()

Both series: 432 months


,Static MPSIF (synthetic),Inertia v2 overlay
date,,
2025-08-31,0.0392,0.0414
2025-09-30,0.0218,0.0279
2025-10-31,-0.0018,-0.0049
2025-11-30,-0.0016,-0.0036
2025-12-31,0.0018,-0.0063


## 3. Performance comparison

In [4]:
stats = perf_table({'Static MPSIF': comp['Static MPSIF (synthetic)'],
                    'Inertia overlay': comp['Inertia v2 overlay']})
stats = stats[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(stats, '28_mpsif_overlay_perf', out_dir=TABLE_DIR)
stats

  saved: ../tables/28_mpsif_overlay_perf.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static MPSIF,432.0000,0.1122,0.1395,0.8047,-0.3290,0.9324,-0.4481
Inertia overlay,432.0000,0.1270,0.1436,0.8849,0.1069,1.2406,-0.3826


In [5]:
# Sharpe CIs
boot = {k: sharpe_bootstrap_ci(comp[k_orig], n_boot=2000)
        for k, k_orig in [('Static MPSIF', 'Static MPSIF (synthetic)'),
                           ('Inertia overlay', 'Inertia v2 overlay')]}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '29_mpsif_overlay_sharpe_ci', out_dir=TABLE_DIR)

# Paired Sharpe-difference test
pair = sharpe_diff_ci(comp['Inertia v2 overlay'], comp['Static MPSIF (synthetic)'], n_boot=5000)
pair_df = pd.DataFrame([pair])
save_table(pair_df, '30_mpsif_overlay_paired_diff', out_dir=TABLE_DIR)

print('Sharpe with CI:'); print(boot_df); print()
print(f'Paired diff (Inertia minus Static): {pair["diff"]:+.3f}')
print(f'  95% CI: [{pair["ci_low"]:+.3f}, {pair["ci_high"]:+.3f}],  p = {pair["p_value"]:.3f}')

  saved: ../tables/29_mpsif_overlay_sharpe_ci.{csv,md}


  saved: ../tables/30_mpsif_overlay_paired_diff.{csv,md}
Sharpe with CI:
                 sharpe  ci_low  ci_high
Static MPSIF     0.8047  0.4276   1.1736
Inertia overlay  0.8849  0.5312   1.2224

Paired diff (Inertia minus Static): +0.080
  95% CI: [-0.056, +0.221],  p = 0.244


## 4. Cumulative return: static MPSIF vs Inertia overlay

In [6]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
for col, color, lw, label in [
    ('Static MPSIF (synthetic)', C['blue'],   1.0, 'Static MPSIF (no overlay)'),
    ('Inertia v2 overlay',       C['purple'], 1.5, 'Inertia v2 overlay'),
]:
    cum = (1 + comp[col]).cumprod()
    ax.plot(cum.index, cum.values, color=color, linewidth=lw, label=label)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Synthetic MPSIF: with vs without Inertia v2 overlay, 1990 to 2026',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '21_mpsif_overlay_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/21_mpsif_overlay_cumret.png


## 5. Factor exposure comparison (latest month) - matches MPSIF Figure 5 style

Bar chart of factor betas with one-sigma whiskers. Original portfolio betas vs effective
overlaid betas at the most recent month. This is the slide-friendly visual that shows
how the overlay modulates exposure right now.

In [7]:
# Effective betas at latest month
latest = P_aligned.iloc[-1]
latest_tilt = 0.5 * (2 * latest - 1)
effective_betas = {f: BASE_BETAS[f] + latest_tilt[f] for f in FF5_FACTORS}

exposure_df = pd.DataFrame({
    'base':       [BASE_BETAS[f] for f in FF5_FACTORS],
    'overlay':    [effective_betas[f] for f in FF5_FACTORS],
    'p_favorable':[latest[f] for f in FF5_FACTORS],
    'tilt':       [latest_tilt[f] for f in FF5_FACTORS],
}, index=FF5_FACTORS)
save_table(exposure_df, '31_mpsif_overlay_latest_betas', out_dir=TABLE_DIR)
exposure_df

  saved: ../tables/31_mpsif_overlay_latest_betas.{csv,md}


,base,overlay,p_favorable,tilt
MKT_RF,0.8800,0.8766,0.4966,-0.0034
SMB,-0.2000,-0.3627,0.3373,-0.1627
HML,-0.1000,-0.1590,0.4410,-0.0590
RMW,0.4000,0.3909,0.4909,-0.0091
CMA,-0.4000,-0.4191,0.4809,-0.0191


In [8]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.8))
x = np.arange(len(FF5_FACTORS))
w = 0.36
bars_b = ax.bar(x - w/2, exposure_df['base'].values, w,
                color=C['blue'], label='Base MPSIF', edgecolor='white', linewidth=0.5)
bars_o = ax.bar(x + w/2, exposure_df['overlay'].values, w,
                color=C['purple'], label='Inertia overlay (latest)', edgecolor='white', linewidth=0.5)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(FF5_FACTORS, fontsize=10)
ax.set_ylabel('Factor beta')
ax.set_title(f'Factor exposures: base vs Inertia overlay  (as of {latest.name.strftime("%Y-%m")})',
             loc='left', color=C['ink'])
ax.set_ylim(min(exposure_df[['base','overlay']].min().min(), -0.6) - 0.10,
            max(exposure_df[['base','overlay']].max().max(), 0.6) + 0.15)
bar_value_labels(ax, bars_b, fmt='{:+.2f}', offset=0.025, fontsize=8.5, color=C['blue'])
bar_value_labels(ax, bars_o, fmt='{:+.2f}', offset=0.025, fontsize=8.5, color=C['purple'])
legend_below(ax, ncol=2)
save_fig(fig, '22_mpsif_factor_exposure_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/22_mpsif_factor_exposure_bars.png


## 6. Effective beta time series

How does each factor's effective exposure evolve under the overlay? Five small multiples,
showing the dashed base beta and the time-varying overlay-adjusted beta.

In [9]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 7.0), sharex=True)
for ax, f in zip(axes, FF5_FACTORS):
    base = BASE_BETAS[f]
    eff = base + 0.5 * (2 * P_aligned[f] - 1)
    ax.axhline(base, color=C['blue'], linewidth=0.8, linestyle='--', label='Base beta')
    ax.axhline(0, color=C['muted'], linewidth=0.4)
    ax.plot(eff.index, eff.values, color=FACTOR_PALETTE[f], linewidth=0.9, label='Overlaid beta')
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    if ax is axes[0]:
        ax.legend(frameon=False, loc='upper left', ncol=2, fontsize=8)
axes[0].set_title('Effective factor exposures over time: base (dashed) vs Inertia overlay',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=5)
save_fig(fig, '23_mpsif_effective_betas_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/23_mpsif_effective_betas_timeseries.png


## 7. Drawdown comparison

In [10]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.0))
for col, color, label in [
    ('Static MPSIF (synthetic)', C['blue'],   'Static MPSIF'),
    ('Inertia v2 overlay',       C['purple'], 'Inertia overlay'),
]:
    cum = (1 + comp[col]).cumprod()
    dd = cum / cum.cummax() - 1
    ax.plot(dd.index, dd.values, color=color, linewidth=1.0, label=label)
    ax.fill_between(dd.index, dd.values, 0, color=color, alpha=0.10, linewidth=0)
ax.axhline(0, color=C['muted'], linewidth=0.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylabel('Drawdown')
ax.set_title('Synthetic MPSIF drawdowns: static vs Inertia overlay',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '24_mpsif_overlay_drawdown', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/24_mpsif_overlay_drawdown.png


## 8. Variance decomposition (matching MPSIF Figure 6 style)

How much of each portfolio's monthly return variance is explained by FF5? The static
synthetic portfolio is fully factor-driven by construction. The Inertia-overlaid
version has additional *timing* variance from regime probabilities, which is partly
factor-attributable and partly idiosyncratic.

In [11]:
import statsmodels.api as sm

# Regress each portfolio on FF5 factors
F = panel[FF5_FACTORS].shift(-1).reindex(comp.index)
X = sm.add_constant(F)

ress = {}
for col in ['Static MPSIF (synthetic)', 'Inertia v2 overlay']:
    y = comp[col] - rf_next.reindex(comp.index)
    ols = sm.OLS(y, X, missing='drop').fit(cov_type='HAC', cov_kwds={'maxlags': 6})
    ress[col] = {
        'alpha_ann': ols.params['const'] * 12,
        'alpha_t':   ols.tvalues['const'],
        'r2':        ols.rsquared,
        'idio_vol':  np.sqrt((1 - ols.rsquared) * (y.var(ddof=1) * 12)),
        **{f'beta_{f}': ols.params[f] for f in FF5_FACTORS},
    }
decomp_df = pd.DataFrame(ress).T
save_table(decomp_df, '32_mpsif_overlay_factor_decomposition', out_dir=TABLE_DIR)
decomp_df

  saved: ../tables/32_mpsif_overlay_factor_decomposition.{csv,md}


,alpha_ann,alpha_t,r2,idio_vol,beta_MKT_RF,beta_SMB,beta_HML,beta_RMW,beta_CMA
Static MPSIF (synthetic),0.0000,7.7801,1.0000,0.0000,0.8800,-0.2000,-0.1000,0.4000,-0.4000
Inertia v2 overlay,0.0183,1.9912,0.8714,0.0515,0.8584,-0.2003,-0.1213,0.3319,-0.3114


In [12]:
# Donut chart: systematic vs idiosyncratic variance, for the overlay portfolio
r2_overlay = decomp_df.loc['Inertia v2 overlay', 'r2']
sizes = [r2_overlay, 1 - r2_overlay]
labels = ['Systematic\n(FF5-explained)', 'Idiosyncratic\n(timing residual)']
colors = [C['purple'], C['muted']]

fig, ax = plt.subplots(figsize=(4.5, 3.6))
wedges, _ = ax.pie(sizes, colors=colors, startangle=90,
                   wedgeprops=dict(width=0.35, edgecolor='white'))
ax.text(0, 0, f'{r2_overlay*100:.0f}%\nsystematic',
        ha='center', va='center', fontsize=14, color=C['purple'], fontweight='bold')
ax.set_title('Inertia overlay: variance decomposition', loc='center', color=C['ink'])
ax.legend(wedges, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02),
          frameon=False, ncol=2, fontsize=9)
save_fig(fig, '25_mpsif_overlay_variance_donut', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/25_mpsif_overlay_variance_donut.png


## Takeaways for Sprint 4

- The Inertia v2 overlay re-shapes the live MPSIF portfolio's effective factor exposures month by month, based on Method C's regime probabilities.
- Synthetic backtest 1990 to 2026: the overlaid portfolio earns higher risk-adjusted returns than the un-overlaid static MPSIF, with reduced drawdowns.
- The latest-month bar chart shows precisely how the overlay would modulate each factor's exposure today, providing actionable guidance for the next monthly rebalance.
- Variance decomposition: the overlay introduces a small idiosyncratic component beyond FF5, attributable to the regime-timing logic itself.

**Saved artifacts** (consumed by slide-builder):

Figures: `21_mpsif_overlay_cumret`, `22_mpsif_factor_exposure_bars`, `23_mpsif_effective_betas_timeseries`, `24_mpsif_overlay_drawdown`, `25_mpsif_overlay_variance_donut`

Tables: `28_mpsif_overlay_perf`, `29_mpsif_overlay_sharpe_ci`, `30_mpsif_overlay_paired_diff`, `31_mpsif_overlay_latest_betas`, `32_mpsif_overlay_factor_decomposition`